# ISLES'24 16-worker persistent downloader

Run **Runtime → Run all**. This notebook continues the existing ISLES'24 download already stored in Google Drive.

It preserves the existing `train.7z.prefix`, completed chunk files, and partial chunk files. It uses **16 parallel workers** and **1 GiB chunks**. If Colab disconnects, reopen this same notebook and run all again.

Before starting new transfers it reports how much data is already persistent. Undersized `.bin` chunks caused by an interrupted Google Drive flush are converted back into resumable partial chunks rather than discarded.


## 1. Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Clone latest repository


## 3. Inspect existing persistent progress


In [ ]:
ARCHIVE = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024/raw/train.7z')
PREFIX = Path(str(ARCHIVE) + '.prefix')
PARTS = Path(str(ARCHIVE) + '.parts')

def gib(n): return n / (1024**3)
saved = 0
if PREFIX.exists():
    saved += PREFIX.stat().st_size
    print(f'Existing prefix: {gib(PREFIX.stat().st_size):.2f} GiB')
if PARTS.exists():
    files = [p for p in PARTS.iterdir() if p.is_file() and p.name != 'manifest.json']
    part_bytes = sum(p.stat().st_size for p in files)
    saved += part_bytes
    complete = sum(p.name.endswith('.bin') for p in files)
    partial = sum(p.name.endswith('.partial') for p in files)
    print(f'Chunk files: {complete} completed-name, {partial} partial; {gib(part_bytes):.2f} GiB stored')
print(f'Total persistent bytes currently visible: ~{gib(saved):.2f} GiB')


In [ ]:
ARCHIVE = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024/raw/train.7z')
PARTS = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024/raw/train.7z.parts')
ARCHIVE.parent.mkdir(parents=True, exist_ok=True)

subprocess.run([
    'python', str(SCRIPT),
    '--archive', str(ARCHIVE),
    '--parts-dir', str(PARTS),
    '--workers','16',
    '--part-size-gib','1',
    '--retries','20',
], check=True)


## 3. Parallel chunk download to persistent Drive


In [ ]:
ARCHIVE = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024/raw/train.7z')
PARTS = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024/raw/train.7z.parts')
ARCHIVE.parent.mkdir(parents=True, exist_ok=True)

subprocess.run([
    'python','scripts/download_isles2024_parallel.py',
    '--archive', str(ARCHIVE),
    '--parts-dir', str(PARTS),
    '--workers','4',
    '--part-size-gib','1',
    '--retries','20',
], check=True)


## 4. Verify persistent result


In [ ]:
print('Archive:', ARCHIVE)
print('Exists:', ARCHIVE.exists())
if ARCHIVE.exists():
    print('Size GiB:', round(ARCHIVE.stat().st_size/(1024**3), 2))
print('Chunks directory exists:', PARTS.exists())
print('Successful completion means the official Zenodo MD5 was verified.')


## Reconnection rule

If Colab disconnects, simply run the notebook again from the top. Do **not** delete the `.parts` directory or `.prefix` file. They are the persistent resume state.
